# 텍스트 특징 추출 (TF-IDF)

### 1. 샘플 코퍼스 연습 

In [1]:
sample_corpus = [
    '자연어처리 강의를 시작하겠습니다.',
    '자연어처리는 재미있습니다.',
    '밥을 먹고 강의를 듣고 있습니다.',
    '이번 자연어처리 강의는 한국어 자연어처리입니다.'
]

In [2]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [3]:
import sklearn
print(sklearn.__version__)

1.8.0


In [4]:
from konlpy.tag import Okt
def my_tokenizer(text):
    return Okt().nouns(text)

In [5]:
# TfidVectorizer 객체 생성 
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(tokenizer=my_tokenizer)
#from sklearn.feature_extraction.text import CountVectorizer
#vectorizer = CountVectorizer(tokenizer=my_tokenizer)
# 특징 집합과 관련 데이터 모델을 생성 
vectorizer.fit(sample_corpus)
print(vectorizer.get_feature_names_out())
# 특징 벡터 추출
sample_dtm = vectorizer.transform(sample_corpus)
type(sample_dtm)
print(sample_dtm.toarray())

c:\Users\user\anaconda3\envs\textmine26\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


['강의' '밥' '시작' '이번' '자연어' '처리' '한국어']
[[0.42817512 0.         0.67081906 0.         0.42817512 0.42817512
  0.        ]
 [0.         0.         0.         0.         0.70710678 0.70710678
  0.        ]
 [0.53802897 0.84292635 0.         0.         0.         0.
  0.        ]
 [0.26813356 0.         0.         0.42008304 0.53626713 0.53626713
  0.42008304]]


### 2. 다음 영화 리뷰로 연습 

In [6]:
import pandas as pd

# 1. CSV 파일 불러오기
df = pd.read_csv(r'D:\Lecture\TextMining_26\data\daum_movie_review.csv', encoding='utf-8') # CSV 파일 경로와 인코딩을 지정하여 데이터프레임으로 불러옵니다. #

# 2. 필요한 컬럼(review)만 선택하고 리스트로 변환 (코퍼스 생성)
# 결측치(NaN)가 있을 경우를 대비해 문자열로 변환하고 제거합니다.
movie_corpus = df['review'].dropna().astype(str).tolist() # 리뷰 컬럼에서 결측치 제거 후 문자열로 변환하여 리스트로 만듦 #

print(f"전체 리뷰 개수: {len(movie_corpus)}개") # 전체 리뷰 개수 출력
print("첫 번째 리뷰 샘플:", movie_corpus[0]) # 첫 번째 리뷰 샘플 출력

전체 리뷰 개수: 14725개
첫 번째 리뷰 샘플: 돈 들인건 티가 나지만 보는 내내 하품만


In [7]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

# 1. 설정 부분 (불용어 및 객체 생성)
my_stop_words = ['영화', '리뷰', '평점'] 
vectorizer = CountVectorizer(
    tokenizer=my_tokenizer, # 사용자 정의 토크나이저 지정
    token_pattern=None, # 사용자 정의 토크나이저를 사용할 때는 token_pattern을 None으로 설정해야 합니다.
    max_features=20,     # 특징 개수를 20개로 상향
    min_df=3, # 최소 3개 이상의 리뷰에 등장하는 단어만 포함
    max_df=0.7, # 전체 리뷰의 70% 이상에 등장하는 단어는 제외
    stop_words=my_stop_words # 사용자 정의 불용어 리스트 추가
)

# 2. 데이터 학습 및 변환 (이 단계가 먼저 와야 합니다)
movie_count_matrix = vectorizer.fit_transform(movie_corpus)

# 3. 결과 확인 및 데이터프레임 변환
features = vectorizer.get_feature_names_out()
dtm_df = pd.DataFrame(movie_count_matrix.toarray(), columns=features)

print(f"최종 추출된 특징 개수: {len(features)}")
display(dtm_df.head()) # 상위 5개 리뷰의 벡터값 확인

# 4. 단어별 총 빈도수 계산 및 확인
# 각 컬럼(단어)의 합계를 구해 내림차순으로 정렬합니다.
word_counts = dtm_df.sum(axis=0).sort_values(ascending=False)

print("\n[단어별 총 빈도수 상위]")
print(word_counts)

최종 추출된 특징 개수: 20


,감동,것,그,눈물,더,마지막,배우,보고,사람,생각,스토리,시간,연기,이,재미,점,정말,좀,진짜,최고
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0



[단어별 총 빈도수 상위]
연기     1293
감동     1223
생각     1027
정말      948
것       927
이       851
배우      847
사람      819
스토리     815
더       791
진짜      781
눈물      737
보고      702
점       692
최고      682
좀       671
재미      641
마지막     612
그       610
시간      596
dtype: int64
